# DWD Wetterdaten Pipeline — Station Regensburg (04104)
**Schritte:**
1. Imports & Konfiguration
2. Download der DWD-Rohdaten
3. ZIPs einlesen & als CSVs speichern
4. CSVs laden, bereinigen & umbenennen
5. Alle Datensätze zusammenführen
6. ML-Preprocessing (Interpolation, KNN, Encoding)
7. Export als `final.csv`

## 1. Imports & Konfiguration

In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sklearn.impute import KNNImputer

# ── Einstellungen ──────────────────────────────────────────
STATION_ID   = "04104"                     # Regensburg
YEARS        = 30                          # Zeitraum in Jahren
DOWNLOAD_DIR = f"downloads_{STATION_ID}"  # Lokaler Ordner für ZIPs
BASE_URL     = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/"

# Parameter die übersprungen werden (kein stündlicher Datensatz)
SKIP_PARAMS  = {'solar/', 'weather_phenomena/'}

END_DATE   = pd.Timestamp.now()
START_DATE = END_DATE - pd.DateOffset(years=YEARS)

print(f"Zeitraum: {START_DATE.date()} bis {END_DATE.date()}")

Zeitraum: 1996-05-19 bis 2026-05-19


## 2. Download der DWD-Rohdaten

In [2]:
def get_links(url):
    """Gibt alle Datei-Links einer DWD-Verzeichnisseite zurück."""
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        return [a['href'] for a in soup.find_all('a', href=True)
                if not a['href'].startswith('../') and a['href'] != '/']
    except requests.RequestException as exc:
        print(f"  Fehler: {url} — {exc}")
        return []


def download_if_missing(file_url, dest_path):
    """Lädt eine Datei herunter, falls sie noch nicht lokal vorhanden ist."""
    if os.path.exists(dest_path):
        return
    print(f"  >>> {os.path.basename(dest_path)}")
    with requests.get(file_url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with open(dest_path, 'wb') as fh:
            for chunk in resp.iter_content(chunk_size=8192):
                fh.write(chunk)


os.makedirs(DOWNLOAD_DIR, exist_ok=True)

for param_dir in get_links(BASE_URL):
    if param_dir not in SKIP_PARAMS and param_dir.endswith('/'):
        param_name = param_dir.rstrip('/')
        dest = os.path.join(DOWNLOAD_DIR, param_name)
        os.makedirs(dest, exist_ok=True)
        print(f"\n{param_name}")
        for sub in ['historical/', 'recent/']:
            for href in get_links(urljoin(BASE_URL, param_dir + sub)):
                if STATION_ID in href and not href.endswith('/'):
                    download_if_missing(
                        urljoin(BASE_URL, param_dir + sub + href),
                        os.path.join(dest, href)
                    )

print("\nDownload abgeschlossen.")


air_temperature
  >>> stundenwerte_TU_04104_19480101_20241231_hist.zip
  >>> stundenwerte_TU_04104_akt.zip

cloud_type
  >>> stundenwerte_CS_04104_19490101_20241231_hist.zip
  >>> stundenwerte_CS_04104_akt.zip

cloudiness
  >>> stundenwerte_N_04104_19490101_20241231_hist.zip
  >>> stundenwerte_N_04104_akt.zip

dew_point
  >>> stundenwerte_TD_04104_19490101_20241231_hist.zip
  >>> stundenwerte_TD_04104_akt.zip

extreme_wind
  >>> stundenwerte_FX_04104_19900701_20241231_hist.zip
  >>> stundenwerte_FX_04104_akt.zip

moisture
  >>> stundenwerte_TF_04104_19490101_20241231_hist.zip
  >>> stundenwerte_TF_04104_akt.zip

precipitation
  >>> stundenwerte_RR_04104_19950901_20241231_hist.zip
  >>> stundenwerte_RR_04104_akt.zip

pressure
  >>> stundenwerte_P0_04104_19490101_20241231_hist.zip
  >>> stundenwerte_P0_04104_akt.zip

soil_temperature
  >>> stundenwerte_EB_04104_19860101_20241231_hist.zip
  >>> stundenwerte_EB_04104_akt.zip

sun
  >>> stundenwerte_SD_04104_19510101_20241231_hist.zip
  >>

## 3. ZIPs einlesen & als CSVs speichern

In [3]:
def read_zip(zip_path):
    """Liest die `produkt_*.txt` aus einem DWD-ZIP und gibt einen DataFrame zurück."""
    with zipfile.ZipFile(zip_path) as z:
        produkt = next((f for f in z.namelist() if f.startswith('produkt_')), None)
        if produkt is None:
            return pd.DataFrame()
        with z.open(produkt) as fh:
            raw = pd.read_csv(fh, sep=';', skipinitialspace=True,
                              low_memory=False, encoding='latin1')
    raw.columns = raw.columns.str.strip()
    raw = raw.drop(columns=[c for c in ['eor', 'STATIONS_ID'] if c in raw.columns], errors='ignore')
    if 'MESS_DATUM' in raw.columns:
        raw['MESS_DATUM'] = pd.to_datetime(raw['MESS_DATUM'], format='%Y%m%d%H', errors='coerce')
    return raw


def build_csv_from_zips(param_name):
    """Liest alle ZIPs eines Parameters, filtert auf 30 Jahre, entfernt Duplikate und -999."""
    param_dir = os.path.join(DOWNLOAD_DIR, param_name)
    zip_files = [os.path.join(param_dir, f) for f in os.listdir(param_dir) if f.endswith('.zip')]
    if not zip_files:
        return

    parts = [read_zip(zf) for zf in zip_files]
    combined = pd.concat(parts, ignore_index=True)

    if combined.empty or 'MESS_DATUM' not in combined.columns:
        return

    combined = (
        combined
        .loc[combined['MESS_DATUM'].between(START_DATE, END_DATE)]
        .sort_values('MESS_DATUM')
        .drop_duplicates(subset=['MESS_DATUM'], keep='last')
        .reset_index(drop=True)
    )

    # DWD-Fehlwert -999 entfernen (nach strip, damit ' -999' auch erwischt wird)
    str_cols = combined.select_dtypes(include=['object', 'str']).columns
    combined[str_cols] = combined[str_cols].apply(lambda col: col.str.strip())
    combined = combined.replace([-999, -999.0, '-999'], pd.NA)

    out = f"wetterdaten_{param_name}_30y.csv"
    combined.to_csv(out, index=False, sep=';')
    print(f"  {param_name:<20} {len(combined):>7} Zeilen  →  {out}")


print("Parameter              Zeilen")
print("-" * 40)
for name in os.listdir(DOWNLOAD_DIR):
    if os.path.isdir(os.path.join(DOWNLOAD_DIR, name)):
        build_csv_from_zips(name)

print("\nAlle CSVs erstellt.")

Parameter              Zeilen
----------------------------------------
  air_temperature       262948 Zeilen  →  wetterdaten_air_temperature_30y.csv
  cloudiness            261789 Zeilen  →  wetterdaten_cloudiness_30y.csv
  cloud_type            261809 Zeilen  →  wetterdaten_cloud_type_30y.csv
  dew_point             262894 Zeilen  →  wetterdaten_dew_point_30y.csv
  extreme_wind          220421 Zeilen  →  wetterdaten_extreme_wind_30y.csv
  moisture              262441 Zeilen  →  wetterdaten_moisture_30y.csv
  precipitation         262436 Zeilen  →  wetterdaten_precipitation_30y.csv
  pressure              262894 Zeilen  →  wetterdaten_pressure_30y.csv
  soil_temperature      223468 Zeilen  →  wetterdaten_soil_temperature_30y.csv
  sun                   194888 Zeilen  →  wetterdaten_sun_30y.csv
  visibility            262135 Zeilen  →  wetterdaten_visibility_30y.csv
  wind                  262554 Zeilen  →  wetterdaten_wind_30y.csv
  wind_synop            262894 Zeilen  →  wetterdaten_w

## 4. CSVs laden, bereinigen & umbenennen

Jeder DWD-Parametername wird auf sinnvolle deutsche Spaltennamen gemappt.
Redundante Spalten (Stations-ID, Qualitätsniveau, Duplikate aus anderen Quellen) werden entfernt.

In [4]:
# Spaltenkonfiguration pro DWD-Parameter:
#   drop   → Spalten die gelöscht werden (Redundanz / Metadaten)
#   rename → DWD-Rohname → sprechender Name

COLUMN_CONFIG = {
    'air_temperature': {
        'drop':   ['QN_9'],
        'rename': {'TT_TU': 'temperatur_c', 'RF_TU': 'rel_feuchte_prozent'}
    },
    'cloud_type': {
        'drop':   ['QN_8', 'V_N_I', 'V_S1_CS', 'V_S2_CS', 'V_S3_CS', 'V_S4_CS'],
        'rename': {
            'V_N':     'bedeckung_gesamt_achtel',
            'V_S1_CSA':'wolkenart_schicht1',   'V_S1_HHS':'wolkenhoehe_schicht1_m', 'V_S1_NS':'bedeckung_schicht1_achtel',
            'V_S2_CSA':'wolkenart_schicht2',   'V_S2_HHS':'wolkenhoehe_schicht2_m', 'V_S2_NS':'bedeckung_schicht2_achtel',
            'V_S3_CSA':'wolkenart_schicht3',   'V_S3_HHS':'wolkenhoehe_schicht3_m', 'V_S3_NS':'bedeckung_schicht3_achtel',
            'V_S4_CSA':'wolkenart_schicht4',   'V_S4_HHS':'wolkenhoehe_schicht4_m', 'V_S4_NS':'bedeckung_schicht4_achtel',
        }
    },
    'cloudiness': {
        'drop':   ['QN_8', 'V_N_I'],
        'rename': {'V_N': 'bedeckung_gesamt_achtel'}
    },
    'sun': {
        'drop':   ['QN_7'],
        'rename': {'SD_SO': 'sonnenscheindauer_min'}
    },
    'visibility': {
        'drop':   ['QN_8', 'V_VV_I'],
        'rename': {'V_VV': 'sichtweite_m'}
    },
    'extreme_wind': {
        'drop':   ['QN_8'],
        'rename': {'FX_911': 'windspitze_max_ms'}
    },
    'wind': {
        'drop':   ['QN_3'],
        'rename': {'F': 'windgeschwindigkeit_ms', 'D': 'windrichtung_grad'}
    },
    'wind_synop': {
        'drop':   ['QN_8'],
        'rename': {'FF': 'windgeschwindigkeit_synop_ms', 'DD': 'windrichtung_synop_grad'}
    },
    'pressure': {
        'drop':   ['QN_8'],
        'rename': {'P': 'luftdruck_nn_hpa', 'P0': 'luftdruck_station_hpa'}
    },
    'dew_point': {
        'drop':   ['QN_8', 'TT'],   # TT ist Duplikat von air_temperature
        'rename': {'TD': 'taupunkt_c'}
    },
    'precipitation': {
        'drop':   ['QN_8', 'WRTR'],
        'rename': {'R1': 'niederschlag_mm', 'RS_IND': 'niederschlag_indikator'}
    },
    'moisture': {
        'drop':   ['QN_8', 'P_STD', 'TT_STD', 'RF_STD', 'TD_STD'],  # Duplikate
        'rename': {'ABSF_STD': 'abs_feuchte_gm3', 'VP_STD': 'dampfdruck_hpa', 'TF_STD': 'feuchttemperatur_c'}
    },
    'soil_temperature': {
        'drop':   ['QN_2', 'V_TE002'],
        'rename': {
            'V_TE005': 'bodentemp_05cm_c', 'V_TE010': 'bodentemp_10cm_c',
            'V_TE020': 'bodentemp_20cm_c', 'V_TE050': 'bodentemp_50cm_c',
            'V_TE100': 'bodentemp_100cm_c'
        }
    },
}

In [5]:
def load_parameter(param_name, config):
    """Lädt eine Parameter-CSV, wendet drop/rename an und setzt den Zeitindex."""
    path = f"wetterdaten_{param_name}_30y.csv"
    if not os.path.exists(path):
        print(f"  Nicht gefunden: {path}")
        return None

    raw = pd.read_csv(path, sep=';', low_memory=False)

    # Spaltennamen und String-Werte trimmen, dann -999 ersetzen
    raw.columns = raw.columns.str.strip()
    str_cols = raw.select_dtypes(include=['object', 'str']).columns
    raw[str_cols] = raw[str_cols].apply(lambda col: col.str.strip())
    raw = raw.replace([-999, -999.0, '-999'], pd.NA)

    # Unwanted columns entfernen und umbenennen
    raw = raw.drop(columns=[c for c in config['drop'] if c in raw.columns])
    raw = raw.rename(columns=config['rename'])
    raw = raw.rename(columns={'MESS_DATUM': 'zeitstempel'})

    raw['zeitstempel'] = pd.to_datetime(raw['zeitstempel'])
    return raw.set_index('zeitstempel')


frames = {}
for param, cfg in COLUMN_CONFIG.items():
    result = load_parameter(param, cfg)
    if result is not None:
        frames[param] = result
        print(f"  {param:<20} geladen  ({result.shape[1]} Spalten, {len(result):,} Zeilen)")

print(f"\n{len(frames)} Parameter geladen.")

  air_temperature      geladen  (2 Spalten, 262,948 Zeilen)
  cloud_type           geladen  (13 Spalten, 261,809 Zeilen)
  cloudiness           geladen  (1 Spalten, 261,789 Zeilen)
  sun                  geladen  (1 Spalten, 194,888 Zeilen)
  visibility           geladen  (1 Spalten, 262,135 Zeilen)
  extreme_wind         geladen  (1 Spalten, 220,421 Zeilen)
  wind                 geladen  (2 Spalten, 262,554 Zeilen)
  wind_synop           geladen  (2 Spalten, 262,894 Zeilen)
  pressure             geladen  (2 Spalten, 262,894 Zeilen)
  dew_point            geladen  (1 Spalten, 262,894 Zeilen)
  precipitation        geladen  (2 Spalten, 262,436 Zeilen)
  moisture             geladen  (3 Spalten, 262,441 Zeilen)
  soil_temperature     geladen  (5 Spalten, 223,468 Zeilen)

13 Parameter geladen.


## 5. Alle Datensätze zusammenführen

Outer-Join über den Zeitindex — fehlende Stunden entstehen wenn ein Sensor
für diesen Zeitpunkt keine Messung lieferte. Das wird in Schritt 6 behandelt.

In [6]:
df = pd.concat(frames.values(), axis=1)
df = df.loc[:, ~df.columns.duplicated()]   # Doppelte Spalten entfernen
df = df.sort_index()

print(f"Kombinierter Datensatz: {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten")
print(f"Zeitraum: {df.index.min()} bis {df.index.max()}")
print(f"\nNaN-Anteil pro Spalte:")
nan_pct = (df.isnull().mean() * 100).round(1)
print(nan_pct[nan_pct > 0].to_string())

Kombinierter Datensatz: 262,948 Zeilen × 35 Spalten
Zeitraum: 1996-05-19 20:00:00 bis 2026-05-18 23:00:00

NaN-Anteil pro Spalte:
temperatur_c                     0.1
rel_feuchte_prozent              0.1
bedeckung_gesamt_achtel          0.4
wolkenart_schicht1              46.6
wolkenhoehe_schicht1_m          19.0
bedeckung_schicht1_achtel       18.9
wolkenart_schicht2              76.9
wolkenhoehe_schicht2_m          60.5
bedeckung_schicht2_achtel       60.5
wolkenart_schicht3              94.8
wolkenhoehe_schicht3_m          85.5
bedeckung_schicht3_achtel       85.4
wolkenart_schicht4              99.4
wolkenhoehe_schicht4_m          99.4
bedeckung_schicht4_achtel       99.4
sonnenscheindauer_min           25.9
sichtweite_m                     0.3
windspitze_max_ms               16.2
windgeschwindigkeit_ms           0.2
windrichtung_grad                0.2
windgeschwindigkeit_synop_ms     0.1
windrichtung_synop_grad          0.1
luftdruck_nn_hpa                 0.1
taupunkt_c         

## 6. ML-Preprocessing

Reihenfolge:
1. **Logisches Auffüllen** — Fehlwerte mit bekanntem Fachsinn
2. **Wolkenkategorien vorbereiten** — `-1` → `keine_wolken`
3. **Zeitinterpolation** — kurzfristige Lücken schließen
4. **KNN-Imputation** — längere Lücken (z.B. Bodentemperatur)
5. **One-Hot-Encoding** — Wolkenarten als Dummy-Variablen
6. **Windrichtung** — zirkulär kodieren (sin/cos)
7. **Zeit-Features** — Stunde, Monat, Tag des Jahres
8. **Finaler Cleanup** — Zeilen ohne Kernwerte löschen

In [7]:
# ── Schritt 1: Logisches Auffüllen ────────────────────────────────────────────
# Fehlender Niederschlag / keine Sonne bedeutet: 0 (nicht gemessen ≠ fehlend)
df['niederschlag_mm']      = df['niederschlag_mm'].fillna(0)
df['sonnenscheindauer_min'] = df['sonnenscheindauer_min'].fillna(0)

# Windspitze: NaN bedeutet keine Böe in dieser Stunde (Ereignis-Spalte, kein Sensor-Ausfall)
# → 0 statt KNN, sonst würden ruhige Stunden falsche Windspitzen bekommen
df['windspitze_max_ms'] = df['windspitze_max_ms'].fillna(0)

# niederschlag_indikator: 0/1 flag — NaN bedeutet kein Ereignis → False
df['niederschlag_indikator'] = (
    df['niederschlag_indikator'].fillna(0).astype(float).astype(bool)
)

# Fehlende Wolkenschicht = keine Wolken in dieser Schicht → 0
cloud_height_cover_cols = [c for c in df.columns if 'schicht' in c and 'art' not in c]
df[cloud_height_cover_cols] = df[cloud_height_cover_cols].fillna(0)

print("Schritt 1 abgeschlossen.")

Schritt 1 abgeschlossen.


In [8]:
# ── Schritt 2: Wolkenkategorien vorbereiten ───────────────────────────────────
# DWD kodiert 'keine Wolken' als -1 — das in einen lesbaren String umwandeln
cloud_type_cols = [c for c in df.columns if 'wolkenart' in c]
for col in cloud_type_cols:
    df[col] = df[col].replace([-1, -1.0, '-1', '-1.0'], 'keine_wolken')

print(f"Wolkenart-Spalten: {cloud_type_cols}")

Wolkenart-Spalten: ['wolkenart_schicht1', 'wolkenart_schicht2', 'wolkenart_schicht3', 'wolkenart_schicht4']


In [9]:
# ── Schritt 3: Zeitinterpolation (nur numerische Spalten) ─────────────────────
num_cols   = df.select_dtypes('number').columns.tolist()
boden_cols = [c for c in num_cols if 'bodentemp' in c]
other_cols = [c for c in num_cols if c not in boden_cols]

# Standard-Sensoren: bis 3 Stunden Lücke schließen
df[other_cols] = df[other_cols].interpolate(method='time', limit=3)

# Bodentemperatur: Messung nur 3× täglich → großzügigeres Limit
df[boden_cols] = df[boden_cols].interpolate(method='time', limit=8)

remaining = df[num_cols].isnull().sum()
print(f"Verbleibende NaNs nach Interpolation:\n{remaining[remaining > 0].to_string()}")

Verbleibende NaNs nach Interpolation:
temperatur_c                      139
rel_feuchte_prozent               141
bedeckung_gesamt_achtel           599
sichtweite_m                      525
windgeschwindigkeit_ms            190
windrichtung_grad                 178
windgeschwindigkeit_synop_ms      214
windrichtung_synop_grad           214
luftdruck_nn_hpa                  163
luftdruck_station_hpa              58
taupunkt_c                        164
abs_feuchte_gm3                   369
dampfdruck_hpa                    369
feuchttemperatur_c                369
bodentemp_05cm_c                 3774
bodentemp_10cm_c                 3770
bodentemp_20cm_c                 3767
bodentemp_50cm_c                 4422
bodentemp_100cm_c               28664


In [10]:
# ── Schritt 4: KNN-Imputation für verbleibende numerische Lücken ──────────────
# Sucht die 5 ähnlichsten Zeilen und nimmt deren gewichteten Mittelwert.
# Läuft in Chunks mit Fortschrittsbalken damit man den Fortschritt sieht.
from tqdm import tqdm
import numpy as np

cols_with_nans = [c for c in num_cols if df[c].isna().any()]

if not cols_with_nans:
    print('Keine NaNs nach Interpolation — KNN nicht nötig.')
else:
    print(f'{len(cols_with_nans)} Spalten mit verbleibenden NaNs:')
    for col in cols_with_nans:
        print(f'  {col:<35} {df[col].isna().sum():>6} NaNs')
    print()

    CHUNK_SIZE = 5000   # Zeilen pro Chunk — größer = schneller aber mehr RAM
    imputer    = KNNImputer(n_neighbors=5, weights='distance')
    result     = df[cols_with_nans].values.copy()
    n_rows     = len(result)

    # Erst einmal komplett fitten (braucht alle Daten als Referenz)
    print('Fitting KNN auf Gesamtdaten ...')
    imputer.fit(result)
    print('Fit abgeschlossen. Transformiere in Chunks ...')

    chunks = range(0, n_rows, CHUNK_SIZE)
    for start in tqdm(chunks, desc='KNN transform', unit='chunk'):
        end = min(start + CHUNK_SIZE, n_rows)
        result[start:end] = imputer.transform(result[start:end])

    df[cols_with_nans] = result
    print('KNN-Imputation abgeschlossen.')

19 Spalten mit verbleibenden NaNs:
  temperatur_c                           139 NaNs
  rel_feuchte_prozent                    141 NaNs
  bedeckung_gesamt_achtel                599 NaNs
  sichtweite_m                           525 NaNs
  windgeschwindigkeit_ms                 190 NaNs
  windrichtung_grad                      178 NaNs
  windgeschwindigkeit_synop_ms           214 NaNs
  windrichtung_synop_grad                214 NaNs
  luftdruck_nn_hpa                       163 NaNs
  luftdruck_station_hpa                   58 NaNs
  taupunkt_c                             164 NaNs
  abs_feuchte_gm3                        369 NaNs
  dampfdruck_hpa                         369 NaNs
  feuchttemperatur_c                     369 NaNs
  bodentemp_05cm_c                      3774 NaNs
  bodentemp_10cm_c                      3770 NaNs
  bodentemp_20cm_c                      3767 NaNs
  bodentemp_50cm_c                      4422 NaNs
  bodentemp_100cm_c                    28664 NaNs

Fitting KNN au

KNN transform: 100%|██████████| 53/53 [03:53<00:00,  4.41s/chunk]

KNN-Imputation abgeschlossen.


In [11]:
# ── Schritt 5: One-Hot-Encoding der Wolkenarten ───────────────────────────────
df = pd.get_dummies(df, columns=cloud_type_cols, dummy_na=False)

print(f"Spalten nach Encoding: {df.shape[1]}")

Spalten nach Encoding: 70


In [12]:
# ── Schritt 6: Windrichtung zirkulär kodieren ─────────────────────────────────
# Windrichtung ist zirkulär (0° = 360°), daher sin/cos statt Rohdegrees
for col in ['windrichtung_grad', 'windrichtung_synop_grad']:
    if col in df.columns:
        rad = df[col].fillna(0) * np.pi / 180
        df[f'{col}_sin'] = np.sin(rad)
        df[f'{col}_cos'] = np.cos(rad)
        df = df.drop(columns=[col])

print("Windrichtung als sin/cos kodiert.")

Windrichtung als sin/cos kodiert.


In [13]:
# ── Schritt 7: Zeit-Features ──────────────────────────────────────────────────
df['stunde']         = df.index.hour
df['monat']          = df.index.month
df['tag_des_jahres'] = df.index.dayofyear

print("Zeitfeatures hinzugefügt: stunde, monat, tag_des_jahres")

Zeitfeatures hinzugefügt: stunde, monat, tag_des_jahres


In [14]:
# ── Schritt 8: Finaler Cleanup ────────────────────────────────────────────────
# Zeilen löschen wo Kern-Sensoren >3h komplett ausgefallen sind
core_cols = [c for c in ['temperatur_c', 'rel_feuchte_prozent'] if c in df.columns]
before    = len(df)
df        = df.dropna(subset=core_cols)
print(f"{before - len(df)} Zeilen mit fehlenden Kernwerten entfernt.")

# verbleibende NaNs (sollte 0 sein)
total_nans = df.isnull().sum().sum()
if total_nans > 0:
    print(f"Warnung: {total_nans} NaNs verbleiben → werden mit 0 gefüllt.")
    df = df.fillna(0)

print(f"\nFinaler Datensatz: {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten")
print(f"NaN-Gesamtanzahl:  {df.isnull().sum().sum()}")
df.info()

0 Zeilen mit fehlenden Kernwerten entfernt.

Finaler Datensatz: 262,948 Zeilen × 75 Spalten
NaN-Gesamtanzahl:  0
<class 'pandas.DataFrame'>
DatetimeIndex: 262948 entries, 1996-05-19 20:00:00 to 2026-05-18 23:00:00
Freq: h
Data columns (total 75 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   temperatur_c                     262948 non-null  float64
 1   rel_feuchte_prozent              262948 non-null  float64
 2   bedeckung_gesamt_achtel          262948 non-null  float64
 3   wolkenhoehe_schicht1_m           262948 non-null  float64
 4   bedeckung_schicht1_achtel        262948 non-null  float64
 5   wolkenhoehe_schicht2_m           262948 non-null  float64
 6   bedeckung_schicht2_achtel        262948 non-null  float64
 7   wolkenhoehe_schicht3_m           262948 non-null  float64
 8   bedeckung_schicht3_achtel        262948 non-null  float64
 9   wolkenhoehe_schicht4_m           262948 non-nul

## 7. Export als `final.parquet`

In [18]:
df.to_parquet('final.parquet', engine='fastparquet', index=True)
print(f"Gespeichert: final.parquet ({df.shape[0]:,} Zeilen × {df.shape[1]} Spalten)")

Gespeichert: final.parquet (262,948 Zeilen × 75 Spalten)


In [19]:
df.head()

,temperatur_c,rel_feuchte_prozent,bedeckung_gesamt_achtel,wolkenhoehe_schicht1_m,bedeckung_schicht1_achtel,wolkenhoehe_schicht2_m,bedeckung_schicht2_achtel,wolkenhoehe_schicht3_m,bedeckung_schicht3_achtel,wolkenhoehe_schicht4_m,...,wolkenart_schicht4_CI,wolkenart_schicht4_CS,wolkenart_schicht4_SC,windrichtung_grad_sin,windrichtung_grad_cos,windrichtung_synop_grad_sin,windrichtung_synop_grad_cos,stunde,monat,tag_des_jahres
zeitstempel,,,,,,,,,,,,,,,,,,,,,
1996-05-19 20:00:00,14.0,83.0,8.0,600.0,5.0,900.0,7.0,0.0,0.0,0.0,...,False,False,False,-0.642788,7.660444e-01,-0.642788,7.660444e-01,20,5,140
1996-05-19 21:00:00,12.9,94.0,8.0,480.0,5.0,780.0,7.0,0.0,0.0,0.0,...,False,False,False,-1.000000,-1.836970e-16,-0.939693,-3.420201e-01,21,5,140
1996-05-19 22:00:00,12.2,96.0,8.0,480.0,5.0,780.0,7.0,0.0,0.0,0.0,...,False,False,False,-0.984808,1.736482e-01,-0.939693,-3.420201e-01,22,5,140
1996-05-19 23:00:00,11.6,96.0,8.0,480.0,3.0,780.0,7.0,0.0,0.0,0.0,...,False,False,False,-1.000000,-1.836970e-16,-1.000000,-1.836970e-16,23,5,140
1996-05-20 00:00:00,11.1,94.0,8.0,480.0,3.0,990.0,7.0,0.0,0.0,0.0,...,False,False,False,-0.984808,1.736482e-01,-0.984808,1.736482e-01,0,5,141


# Datensatzbeschreibung: `final.parquet`
**DWD Station Regensburg — Station ID 04104**

---

## Überblick

| Eigenschaft | Wert |
| :--- | :--- |
| Quelle | Deutscher Wetterdienst (DWD) Open Data |
| Station | Regensburg (ID 04104) |
| Zeitraum | 1996-05-18 bis 2026-05-17 (30 Jahre) |
| Auflösung | Stündlich |
| Zeilen | ~262.000 |
| Spalten | ~79 |
| Index | `zeitstempel` (DatetimeIndex) |

### Vorverarbeitung
- **Fehlwerte:** DWD-Sentinel `-999` wurde zu `NaN` konvertiert
- **Interpolation:** Kurzfristige Lücken (<3h) per Zeitinterpolation geschlossen; Bodentemperatur (<8h)
- **KNN-Imputation:** Verbleibende Lücken in Sensorspalten mit `KNNImputer(n_neighbors=5)` gefüllt
- **Ereignisspalten** (`niederschlag_mm`, `sonnenscheindauer_min`, `windspitze_max_ms`): `NaN` → `0` (kein Ereignis, kein Sensor-Ausfall)
- **Windrichtung:** Zirkulär kodiert (sin/cos) zur Vermeidung von Sprüngen an der 360°-Grenze
- **Wolkenarten:** One-Hot-kodiert nach WMO-Wolkengattungen

### Bekannte Einschränkungen
- `niederschlag_indikator = False` bedeutet entweder "kein Regen" **oder** "fehlende Messung" — beide Fälle sind nach dem Join nicht mehr unterscheidbar
- `windspitze_max_ms = 0` bedeutet entweder "keine Böe gemessen" **oder** seltener "Sensor-Ausfall" (-999) — nicht trennbar nach dem Join
- `bodentemp_*`-Spalten sind vor ca. 2000 vollständig KNN-imputiert (Sensor erst ab ~1996/2000 aktiv) — für Zeitreihenanalysen vor 2000 mit Vorsicht verwenden
- `bodentemp_100cm_c` hat den höchsten Imputationsanteil (~28.000 Stunden)

---

## 1. Zeitvariablen

| Spaltenname | Beschreibung | Typ |
| :--- | :--- | :--- |
| `zeitstempel` | **Index:** Datum und Uhrzeit der Messung | `DatetimeIndex` |
| `stunde` | Stunde der Messung (aus Zeitstempel) | `int` (0–23) |
| `monat` | Monat der Messung (aus Zeitstempel) | `int` (1–12) |
| `tag_des_jahres` | Tag des Jahres (aus Zeitstempel) | `int` (1–366) |

---

## 2. Temperatur & Feuchtigkeit

| Spaltenname | Beschreibung | Einheit |
| :--- | :--- | :--- |
| `temperatur_c` | Lufttemperatur in 2 m Höhe | `°C` |
| `taupunkt_c` | Taupunkttemperatur in 2 m Höhe | `°C` |
| `feuchttemperatur_c` | Berechnete Feuchttemperatur (Verdunstungskühlung) | `°C` |
| `rel_feuchte_prozent` | Relative Luftfeuchtigkeit | `%` |
| `abs_feuchte_gm3` | Absolute Feuchte | `g/m³` |
| `dampfdruck_hpa` | Partialdruck des Wasserdampfs | `hPa` |

---

## 3. Bodentemperaturen

| Spaltenname | Beschreibung | Einheit |
| :--- | :--- | :--- |
| `bodentemp_05cm_c` | Erdbodentemperatur in 5 cm Tiefe | `°C` |
| `bodentemp_10cm_c` | Erdbodentemperatur in 10 cm Tiefe | `°C` |
| `bodentemp_20cm_c` | Erdbodentemperatur in 20 cm Tiefe | `°C` |
| `bodentemp_50cm_c` | Erdbodentemperatur in 50 cm Tiefe | `°C` |
| `bodentemp_100cm_c` | Erdbodentemperatur in 100 cm Tiefe | `°C` |

---

## 4. Luftdruck & Sichtweite

| Spaltenname | Beschreibung | Einheit |
| :--- | :--- | :--- |
| `luftdruck_nn_hpa` | Luftdruck reduziert auf Meereshöhe (Normalnull) | `hPa` |
| `luftdruck_station_hpa` | Luftdruck auf Stationshöhe (unkorrigiert) | `hPa` |
| `sichtweite_m` | Meteorologische Sichtweite | `m` |

---

## 5. Wind

> **Hinweis:** Windrichtungen wurden zirkulär kodiert (Sinus/Kosinus), da 0° und 360° identisch sind und ein lineares Modell sonst einen Sprung lernen würde.

> **Hinweis:** `windspitze_max_ms = 0` bedeutet entweder keine Böe gemessen oder (selten) Sensor-Ausfall. Beide Fälle sind nicht unterscheidbar.

| Spaltenname | Beschreibung | Einheit |
| :--- | :--- | :--- |
| `windgeschwindigkeit_ms` | Stundenmittel der Windgeschwindigkeit | `m/s` |
| `windgeschwindigkeit_synop_ms` | Synoptische Messung der Windgeschwindigkeit | `m/s` |
| `windspitze_max_ms` | Maximale Windböe innerhalb der Stunde; 0 = keine Böe | `m/s` |
| `windrichtung_grad_sin` | Sinus der Windrichtung (Stundenmittel) | `float` (−1 bis 1) |
| `windrichtung_grad_cos` | Kosinus der Windrichtung (Stundenmittel) | `float` (−1 bis 1) |
| `windrichtung_synop_grad_sin` | Sinus der synoptischen Windrichtung | `float` (−1 bis 1) |
| `windrichtung_synop_grad_cos` | Kosinus der synoptischen Windrichtung | `float` (−1 bis 1) |

---

## 6. Niederschlag & Sonnenschein

> **Hinweis:** `niederschlag_indikator = False` bedeutet entweder kein Niederschlagsereignis **oder** fehlende Messung — nach dem Datenbankjoin nicht mehr unterscheidbar.

| Spaltenname | Beschreibung | Einheit |
| :--- | :--- | :--- |
| `niederschlag_mm` | Niederschlagshöhe (Stundensumme); 0 = kein Niederschlag | `mm` |
| `niederschlag_indikator` | Ob ein Niederschlagsereignis stattfand | `bool` |
| `sonnenscheindauer_min` | Sonnenscheindauer (Stundensumme); 0 = keine Sonne | `min` |

---

## 7. Bewölkung

> **Hinweis:** Wolkenarten wurden One-Hot-kodiert. Suffixe entsprechen den WMO-Wolkengattungen. `keine_wolken` ist eine eigene Kategorie (DWD-Code `-1`). Schichten 2–4 sind `0` wenn keine weitere Schicht vorhanden war.

### Bedeckung & Höhe

| Spaltenname | Beschreibung | Einheit |
| :--- | :--- | :--- |
| `bedeckung_gesamt_achtel` | Gesamtbedeckungsgrad des Himmels | `0–8 Achtel` |
| `bedeckung_schicht[1-4]_achtel` | Bedeckungsgrad der jeweiligen Wolkenschicht | `0–8 Achtel` |
| `wolkenhoehe_schicht[1-4]_m` | Untergrenze der jeweiligen Wolkenschicht | `m` |

### Wolkenarten (One-Hot)

Jede Schicht (1–4) hat folgende Bool-Spalten:

| Suffix | WMO-Gattung |
| :--- | :--- |
| `_AC` | Altocumulus |
| `_AS` | Altostratus |
| `_CB` | Cumulonimbus |
| `_CC` | Cirrocumulus |
| `_CI` | Cirrus |
| `_CS` | Cirrostratus |
| `_CU` | Cumulus |
| `_NS` | Nimbostratus |
| `_SC` | Stratocumulus |
| `_ST` | Stratus |
| `_keine_wolken` | Keine Wolken in dieser Schicht (DWD-Code `-1`) |

Beispiel: `wolkenart_schicht1_CU = True` bedeutet Cumulus in der untersten Schicht.